# DINO — Emerging Properties in Self-Supervised Vision Transformers (2021)

_Papers · Self-supervised & Representation_

**A student matches a momentum-teacher's output distribution with no labels; centering + sharpening stop collapse.**

---

This notebook is a practice scaffold for the **DINO — Emerging Properties in Self-Supervised Vision Transformers (2021)** lesson. We assemble the DINO self-distillation loop one piece at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Sanity-check the DINO loss by hand

DINO's loss is a cross-entropy `H(P_t, P_s)` between a teacher distribution `P_t` and a student distribution `P_s`. The teacher target is built by **centering** (subtract `c`) then **sharpening** (divide by a small temperature `τ_t`) before softmax. Working one example through first shows the key identity `H = h(P_t) + KL(P_t‖P_s)`: the loss floor is the teacher's own entropy, so the student is driving toward matching the teacher exactly.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity-check the worked example: teacher target P_t, loss H, and decomposition (Eqns. 1,2,5).
gt = torch.tensor([2.0, 1.0, 0.0, -1.0])      # teacher logits, K=4
c  = torch.tensor([0.5, 0.5, 0.0, 0.0])       # center
tau_t, tau_s = 0.5, 1.0

Pt = F.softmax((gt - c) / tau_t, dim=0)        # center -> sharpen -> softmax  (Eqn. 1)
gs = torch.tensor([1.0, 0.5, 0.0, 0.0])       # student logits (other view)
Ps = F.softmax(gs / tau_s, dim=0)

H   = -(Pt * Ps.log()).sum()                   # cross-entropy  (Eqn. 2)
hPt = -(Pt * Pt.log()).sum()                   # teacher entropy
KL  = (Pt * (Pt.log() - Ps.log())).sum()       # D_KL(Pt || Ps)

print("Pt =", [round(v,4) for v in Pt.tolist()], " Ps =", [round(v,4) for v in Ps.tolist()])
print("H(Pt,Ps) =", round(H.item(),4), " h(Pt) =", round(hPt.item(),4),
      " KL =", round(KL.item(),4), " h+KL =", round((hPt+KL).item(),4), " lnK =", round(np.log(4),4))
# Pt = [0.839, 0.1135, 0.0418, 0.0057]  Ps = [0.4269, 0.2589, 0.1571, 0.1571]
# H(Pt,Ps) = 0.9553  h(Pt) = 0.5562  KL = 0.3991  h+KL = 0.9553  lnK = 1.3863

### Step 2 — Build the encoder and projection head

DINO has no class labels, so the network outputs `K` arbitrary "prototype" logits (here `K = 256`, **not** the 10 digit classes). A small conv encoder `f` produces a feature vector, and a projection head maps it to the `K` logits. These are the shared building blocks for both the student and the teacher.

In [ ]:
# Building blocks: encoder f + projection head (-> K logits) from nn primitives.
K = 256                                        # output dim of the head (NOT class labels)

class Encoder(nn.Module):
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)

    def forward(self, x):
        return F.relu(self.fc(self.net(x)))

def head(fin=64, hid=128, out=K):              # projection head -> K logits
    return nn.Sequential(nn.Linear(fin, hid), nn.GELU(), nn.Linear(hid, out))

### Step 3 — Define the student/teacher DINO pair

The student is trained by gradient descent; the teacher is an **EMA** (exponential moving average) copy with gradients turned off. The teacher's output is centered and sharpened, the student's is just softened. The loss is symmetric — each view's teacher target supervises the **other** view's student — and after every step we update the running center and the teacher weights. `teacher_entropy` is a collapse monitor: if it crashes toward zero, the teacher is emitting the same distribution for every image.

In [ ]:
# The DINO pair: student (trained) + teacher (EMA, stop-grad, centered+sharpened).
class DINO(nn.Module):
    def __init__(self, use_centering=True, tau_s=0.1, tau_t=0.04, lam=0.99, m=0.9):
        super().__init__()
        self.tau_s, self.tau_t, self.lam, self.m = tau_s, tau_t, lam, m
        self.use_centering = use_centering
        self.enc_s, self.hd_s = Encoder(), head()                 # student
        self.enc_t = copy.deepcopy(self.enc_s)                    # teacher = EMA copy (no grad)
        self.hd_t  = copy.deepcopy(self.hd_s)
        for p in list(self.enc_t.parameters()) + list(self.hd_t.parameters()):
            p.requires_grad_(False)
        self.register_buffer("center", torch.zeros(K))           # the center c (Eqn. 4)

    def student(self, x):                       # logits -> log P_s (softmax @ tau_s, Eqn. 1)
        return F.log_softmax(self.hd_s(self.enc_s(x)) / self.tau_s, dim=1)   # log P_s

    @torch.no_grad()
    def teacher(self, x):                       # center -> sharpen -> softmax -> P_t (stop-gradient)
        g = self.hd_t(self.enc_t(x))
        if self.use_centering:
            g = g - self.center
        return F.softmax(g / self.tau_t, dim=1), g                # return P_t and raw logits for center

    def loss_and_update_center(self, v1, v2):   # symmetrized cross-entropy (Eqn. 2)
        Pt1, g1 = self.teacher(v1)
        Pt2, g2 = self.teacher(v2)
        logPs1, logPs2 = self.student(v1), self.student(v2)
        # teacher of one view supervises student of the OTHER view (cross)
        loss = -(Pt1 * logPs2).sum(1).mean() - (Pt2 * logPs1).sum(1).mean()
        with torch.no_grad():                   # center update (Eqn. 4): EMA of batch-mean teacher logits
            batch_mean = torch.cat([g1, g2], 0).mean(0)
            self.center.mul_(self.m).add_(batch_mean, alpha=1 - self.m)
        return loss

    @torch.no_grad()
    def ema_update(self):                       # teacher EMA: theta_t <- lam*theta_t + (1-lam)*theta_s
        for s, t in [(self.enc_s, self.enc_t), (self.hd_s, self.hd_t)]:
            for ps, pt in zip(s.parameters(), t.parameters()):
                pt.mul_(self.lam).add_(ps.detach(), alpha=1 - self.lam)

    @torch.no_grad()
    def teacher_entropy(self, X):               # mean per-image entropy of P_t (collapse monitor)
        Pt, _ = self.teacher(X)
        return -(Pt * (Pt + 1e-9).log()).sum(1).mean().item()

### Step 4 — Two-view augmentation, unlabelled data, and the pretraining loop

DINO learns by making two random augmentations of each image agree. We build an augmentation pipeline, take an **unlabelled** MNIST subset (the labels are kept aside only for a probe later), and write the pretraining loop. Each step computes the loss, takes a student gradient step, **then** updates the EMA teacher — the order matters.

In [ ]:
# Two-view augmentation + an UNLABELLED MNIST subset (torchvision, preinstalled).
aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])     # used ONLY for the probe later

def pretrain(use_centering=True, epochs=15):
    torch.manual_seed(0)
    m = DINO(use_centering=use_centering).to(device)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    m.train()
    B = 128
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs))
        tot = 0.0
        nb = 0
        for s in range(0, len(imgs), B):
            bi = perm[s:s + B]
            v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)
            loss = m.loss_and_update_center(v1, v2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            m.ema_update()                      # student step THEN EMA
            tot += loss.item()
            nb += 1
        if ep % 3 == 0:
            Xs = torch.stack([base(im) for im in imgs[:500]]).to(device)
            print(f"  pretrain ep {ep}  loss {tot/nb:.4f}  teacher entropy {m.teacher_entropy(Xs):.4f}")
    return m

print("\n=== full DINO (centering + sharpening) ===")
m = pretrain(use_centering=True)

### Step 5 — Freeze the encoder and run the linear-evaluation protocol

To test whether DINO learned useful features without labels, we **freeze** the student encoder and extract features. Then a linear probe (a single linear layer on the frozen features) is trained on a handful of labels and compared against a fresh conv net trained end-to-end on the same few labels. If the probe wins in the low-label regime, the label-free pretraining transferred real structure.

In [ ]:
# FREEZE the student encoder, extract features (linear-evaluation protocol).
def features(model):
    model.eval()
    with torch.no_grad():
        return model.enc_s(torch.stack([base(im) for im in imgs]).to(device)).cpu()

feats = features(m)
print("feature std across images (full DINO):", round(feats.std(0).mean().item(), 4), "(healthy: > 0)")

def linear_probe(feats, n_lab):                 # train ONLY a linear classifier on frozen features
    accs = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10)
        o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad()
            F.cross_entropy(clf(feats[tr]), labels[tr]).backward()
            o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

def from_scratch(n_lab):                         # train a fresh conv net end-to-end on the few labels
    accs = []
    for seed in range(3):
        torch.manual_seed(seed)
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64, 10))
        o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr])
        net.train()
        for _ in range(60):
            o.zero_grad()
            F.cross_entropy(net(Xtr), labels[tr]).backward()
            o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

print("\nlabels | probe(frozen DINO) | from-scratch")
for n in [20, 50, 100, 300]:
    print(f"{n:6d} |       {linear_probe(feats, n):.3f}       |    {from_scratch(n):.3f}")

### Step 6 — Ablation: remove centering and watch the teacher collapse

Centering is DINO's defense against collapse. Re-pretrain with centering **off**: sharpening alone now drives the teacher to emit the same near-one-hot distribution for every image, so its per-image entropy crashes toward zero (the paper's Sec 5.3 collapse). The contrast in teacher entropy — healthy vs near-zero — is the decisive signal that centering is what keeps the teacher informative.

In [ ]:
# ABLATION: remove CENTERING -> teacher distribution collapses onto one dimension.
print("\n=== ablation: NO centering (expect teacher collapse) ===")
m_ab = pretrain(use_centering=False)
feats_ab = features(m_ab)

Xs = torch.stack([base(im) for im in imgs[:500]]).to(device)
print("teacher entropy  full vs no-centering:", round(m.teacher_entropy(Xs), 4),
      "vs", round(m_ab.teacher_entropy(Xs), 4), "(collapsed: ~0)")
print("probe(no-centering DINO) @100 labels:", round(linear_probe(feats_ab, 100), 3))
# Full DINO: probe beats from-scratch at every budget; teacher entropy stays healthy (~4.7 over K=256).
# No centering: teacher entropy collapses toward 0 -- it emits the SAME near-one-hot distribution for
#   every image (one dimension dominates), exactly the paper's Sec 5.3 collapse. At this tiny scale the
#   frozen backbone probe degrades only mildly in 15 epochs, but the teacher-distribution collapse -- the
#   thing centering exists to prevent -- is unambiguous (entropy ~4.7 -> ~0).
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does DINO learn useful features with NO labels — and does removing CENTERING collapse the teacher?_

### Step 7 — Rebuild the DINO model self-contained

This visualization cell stands alone, so it re-imports everything and re-defines the encoder, head, and DINO module exactly as above. Re-running the full pipeline here keeps the panel reproducible in isolation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

# DINO with NO labels on UNLABELLED MNIST: freeze, probe vs from-scratch, and the
# no-centering collapse ablation (toy reproduction of paper Section 5.3).
K = 256

class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Linear(32, feat)
    def forward(s, x):
        return F.relu(s.fc(s.net(x)))

def head(fin=64, hid=128, out=K):
    return nn.Sequential(nn.Linear(fin,hid), nn.GELU(), nn.Linear(hid,out))

class DINO(nn.Module):
    def __init__(s, use_centering=True, tau_s=0.1, tau_t=0.04, lam=0.99, m=0.9):
        super().__init__()
        s.tau_s, s.tau_t, s.lam, s.m, s.use_centering = tau_s, tau_t, lam, m, use_centering
        s.enc_s, s.hd_s = Encoder(), head()
        s.enc_t, s.hd_t = copy.deepcopy(s.enc_s), copy.deepcopy(s.hd_s)
        for p in list(s.enc_t.parameters()) + list(s.hd_t.parameters()):
            p.requires_grad_(False)
        s.register_buffer("center", torch.zeros(K))
    def student(s, x):
        return F.log_softmax(s.hd_s(s.enc_s(x))/s.tau_s, dim=1)
    @torch.no_grad()
    def teacher(s, x):
        g = s.hd_t(s.enc_t(x))
        if s.use_centering:
            g = g - s.center
        return F.softmax(g/s.tau_t, dim=1), g
    def step_loss(s, v1, v2):
        Pt1, g1 = s.teacher(v1)
        Pt2, g2 = s.teacher(v2)
        L = -(Pt1*s.student(v2)).sum(1).mean() - (Pt2*s.student(v1)).sum(1).mean()
        with torch.no_grad():
            s.center.mul_(s.m).add_(torch.cat([g1,g2],0).mean(0), alpha=1-s.m)
        return L
    @torch.no_grad()
    def ema(s):
        for a,b in [(s.enc_s,s.enc_t),(s.hd_s,s.hd_t)]:
            for ps,pt in zip(a.parameters(), b.parameters()):
                pt.mul_(s.lam).add_(ps.detach(), alpha=1-s.lam)
    @torch.no_grad()
    def tent(s, X):
        Pt,_ = s.teacher(X)
        return -(Pt*(Pt+1e-9).log()).sum(1).mean().item()

### Step 8 — Pretrain (with and without centering) on the unlabelled subset

Rebuild the augmentation pipeline and unlabelled MNIST subset, then define `pretrain`, `features`, the linear `probe`, and the `scratch` baseline. These mirror Steps 4–5; running them sets up both the full-DINO model and the no-centering ablation for the final comparison.

In [ ]:
aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5,1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])

def pretrain(use_centering=True, epochs=15):
    torch.manual_seed(0)
    m = DINO(use_centering=use_centering)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=1e-3)
    m.train()
    B = 128
    for ep in range(epochs):
        perm = np.random.permutation(len(imgs))
        for s0 in range(0, len(imgs), B):
            bi = perm[s0:s0+B]
            v1 = torch.stack([aug(imgs[i]) for i in bi])
            v2 = torch.stack([aug(imgs[i]) for i in bi])
            loss = m.step_loss(v1, v2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            m.ema()
    return m

def features(m):
    m.eval()
    with torch.no_grad():
        return m.enc_s(torch.stack([base(im) for im in imgs]))

def probe(feats, n):
    a = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n]
        te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10)
        o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad()
            F.cross_entropy(clf(feats[tr]), labels[tr]).backward()
            o.step()
        with torch.no_grad():
            a.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return round(float(np.mean(a)), 3)

def scratch(n):
    a = []
    for seed in range(3):
        torch.manual_seed(seed)
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n]
        te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64,10))
        o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr])
        net.train()
        for _ in range(60):
            o.zero_grad()
            F.cross_entropy(net(Xtr), labels[tr]).backward()
            o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            a.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return round(float(np.mean(a)), 3)

### Step 9 — Compare probe vs from-scratch, then run the collapse ablation

Pretrain full DINO, then print the probe-vs-from-scratch accuracy at each label budget and the healthy teacher entropy. Finally, re-pretrain without centering and print its teacher entropy — collapsing toward zero — alongside its probe score. This is the headline panel: label-free features that beat from-scratch, and the collapse that centering prevents.

In [ ]:
m_full = pretrain(use_centering=True)
f_full = features(m_full)
for n in [20,50,100,300]:
    print(n, "probe", probe(f_full, n), "scratch", scratch(n))

Xs = torch.stack([base(im) for im in imgs[:500]])
print("full DINO teacher entropy:", round(m_full.tent(Xs),3))

m_ab = pretrain(use_centering=False)
f_ab = features(m_ab)
print("no-centering teacher entropy:", round(m_ab.tent(Xs),3), "(collapsed ~0)")
print("no-centering probe @100:", probe(f_ab, 100))
# Full DINO probe > from-scratch at every budget (no labels used).
# No centering -> teacher entropy collapses toward 0 (one dim dominates) -- the paper's Sec 5.3 collapse.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline (no labels). You pretrained a small encoder with DINO on unlabelled images —
            no labels, no negatives, just student-matches-teacher cross-entropy — froze it, and trained
            a linear probe on just 20 labels; you also trained a from-scratch model on the same 20 labels. The
            probe scores much higher. What does that demonstrate, and how does DINO's collapse defense differ
            from BYOL's (paper-byol)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the two accuracies at the smallest label budget; the frozen-DINO probe wins. — _With only 20 labels a from-scratch net has too little signal; the probe inherits features learned from thousands of unlabelled images._
- Note the DINO loss had no labels and no negatives — only cross-entropy to a centered, sharpened EMA teacher. — _The structure came from self-distillation, not supervision._
- Contrast the anti-collapse mechanism: DINO uses centering + sharpening on the teacher; BYOL uses an asymmetric predictor + stop-grad. — _Both avoid the constant-output collapse without negatives, but by different means — DINO operates on the teacher's output distribution, BYOL on the network architecture._

**Answer:** It demonstrates that label-free self-distillation transfers: in the low-label regime
                 a linear probe on frozen DINO features beats from-scratch because the features were shaped by
                 thousands of unlabelled images via cross-entropy to a momentum teacher. DINO avoids collapse
                 with centering + sharpening of the teacher distribution (Eqns. 1, 4), whereas BYOL uses
                 an asymmetric predictor head + stop-gradient — same goal (no constant collapse, no
                 negatives), different mechanism. Our CODEVIZ panel shows the probe beating from-scratch across
                 the label budgets.

</details>

**Problem 2.** Your worked example gave teacher target $P_t=[0.839,0.114,0.042,0.006]$, student
            $P_s=[0.427,0.259,0.157,0.157]$, and loss $H(P_t,P_s)=0.9553$. Suppose the student improves so its
            distribution becomes $P_s'=[0.80,0.13,0.04,0.03]$ (much closer to the teacher). Does the loss go up
            or down, and roughly toward what floor?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $H(P_t,P_s') = -\sum_k P_t^{(k)}\log P_s'^{(k)}$ with the new $P_s'$. — _Cross-entropy is smallest when the student matches the teacher._
- $H \approx -(0.839\ln 0.80 + 0.114\ln 0.13 + 0.042\ln 0.04 + 0.006\ln 0.03) \approx 0.62$. — _Mass now sits where the teacher's mass is, so $-\log P_s$ is small exactly where $P_t$ is large._
- Compare to the floor: the smallest possible $H$ here is the teacher entropy $h(P_t)=0.5562$ (when $P_s=P_t$ exactly, the KL term is $0$). — _By Eqn. 5, $H=h(P_t)+D_{KL}$, and $D_{KL}\ge 0$ with equality iff $P_s=P_t$._

**Answer:** The loss goes down, from $0.9553$ to about $0.62$, because the student distribution
                 now puts its mass where the teacher's is (the KL term shrank). It cannot fall below the
                 teacher's own entropy $h(P_t)=0.5562$: by Eqn. 5, $H=h(P_t)+D_{KL}(P_t\Vert P_s)$ and the KL
                 term is $\ge 0$, hitting $0$ only when $P_s=P_t$ exactly. So $h(P_t)$ is the floor the student
                 is driving toward.

</details>

**Problem 3.** Ablation (the assigned one). You remove centering (keep sharpening at small $\tau_t$)
            and re-pretrain DINO. The training loss drops fast and the teacher's mean per-image output entropy
            crashes from ~4.7 (spread over $K$ slots) toward $0$. What happened to the teacher's output, and
            which design choice prevents it in real DINO?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Inspect the teacher distribution $P_t$ across different images after the no-centering run. — _They are nearly identical and peaked on the same single dimension for every image — the one-dimension collapse._
- See why the loss fell: a degenerate, near-constant teacher is trivial for the student to match, so cross-entropy (and the KL term) drop without learning anything useful. — _With no negatives and no centering, sharpening alone drives the teacher to a dominant dimension; nothing forces different images to differ._
- Restore centering: subtract the running center $c$ (Eqn. 4) from teacher logits before the softmax. — _Centering removes the component common to all images, stopping any one dimension from dominating — it balances sharpening (paper §5.3, Figure 6)._

**Answer:** The teacher distribution collapsed onto one dimension: with centering gone, sharpening
                 alone makes the teacher emit the same near-one-hot distribution for every image, so the
                 student matches a target that carries no information — the per-image entropy crashes toward $0$
                 and the loss falls without learning. Real DINO prevents this with centering (Eqn. 4):
                 subtracting the running mean $c$ of teacher logits removes what is common across the batch,
                 balancing sharpening so the teacher stays informative (paper §5.3). Our CODEVIZ panel shows the
                 teacher's mean per-image entropy crushed from ~4.7 toward ~0 in the no-centering run versus the
                 healthy value with centering on. (At full scale this teacher collapse drags the learned
                 features down too; in our tiny short run the clean, decisive signal is the teacher's
                 entropy.)

</details>